In [1]:
from pathlib import Path
import json
import tensorflow as tf

INPUT_DIR = Path("/kaggle/input")


def find_one(filename):
    matches = list(INPUT_DIR.rglob(filename))

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one {filename}, found {len(matches)}: {matches}"
        )

    return matches[0]


gpus = tf.config.list_physical_devices("GPU")

print("Current TensorFlow:", tf.__version__)
print("Detected GPUs:", gpus)

if not gpus:
    raise RuntimeError("GPU unavailable. Do not start baseline training.")

config_path = find_one("run_config.json")
original_config = json.loads(
    config_path.read_text(encoding="utf-8")
)

print("\nOriginal run configuration:")
print(json.dumps(original_config, indent=2))

print("\nSaved checkpoint:", find_one("best_model.keras"))
print("Training manifest:", find_one("train.csv"))
print("Validation manifest:", find_one("validation.csv"))
print("\nSetup inspection complete. Training has not started.")

Current TensorFlow: 2.20.0
Detected GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]

Original run configuration:
{
  "tensorflow_version": "2.20.0",
  "seed": 42,
  "max_epochs": 30,
  "batch_size": 32,
  "image_shape": [
    224,
    224,
    1
  ],
  "training_images": 23873,
  "validation_images": 6103,
  "class_mapping": {
    "negative": 0,
    "positive": 1
  },
  "class_weights": {
    "0": 1.0669020378977476,
    "1": 0.9409932991722507
  },
  "checkpoint_monitor": "val_loss",
  "augmentation": {
    "rotation_factor": 0.02,
    "translation_height": 0.03,
    "translation_width": 0.03
  }
}

Saved checkpoint: /kaggle/input/datasets/mamtakumaridxcv/covidx-trained-model-backup/20261018T170010_055087Z/best_model.keras
Training manifest: /kaggle/input/datasets/mamtakumaridxcv/covidx-audit-backup/artifacts/manifests/train.csv
Validation manifest: /kaggle/input/datasets/mamtakumaridxcv/covidx-

In [2]:
import hashlib
import sys
import pandas as pd

dataset_dir = find_one("train.txt").parent

expected_hashes = {
    "train.txt": (
        "52af9a4fdc769396bc5ce530872cf0d51bce"
        "8fd7e25579b333007d4ebee6a6ba"
    ),
    "test.txt": (
        "7c92ae0bef49652005e1541a1f2b774e31"
        "3be4431a891cb0aa5539d37ca2a71e"
    ),
}

for filename, expected in expected_hashes.items():
    actual = hashlib.sha256(
        (dataset_dir / filename).read_bytes()
    ).hexdigest()
    assert actual == expected, f"Dataset mismatch: {filename}"

training = pd.read_csv(
    find_one("train.csv"), dtype=str, keep_default_na=False
)
validation = pd.read_csv(
    find_one("validation.csv"), dtype=str, keep_default_na=False
)

for name, table, count in [
    ("Training", training, 23873),
    ("Validation", validation, 6103),
]:
    assert len(table) == count
    assert table["filename"].is_unique
    assert table["sha256"].is_unique
    assert set(table["label"]) == {"negative", "positive"}

    missing = [
        filename for filename in table["filename"]
        if not (dataset_dir / "train" / filename).is_file()
    ]
    assert not missing, f"{name}: missing images: {missing[:5]}"

for column in ["filename", "sha256", "split_group"]:
    assert set(training[column]).isdisjoint(validation[column])

model_files = list(INPUT_DIR.glob("**/src/covid_xray/model.py"))
assert len(model_files) == 1, model_files

sys.path.insert(0, str(model_files[0].parents[1]))

from covid_xray.data import make_dataset, LABEL_MAP
from covid_xray.model import build_model

print("Dataset checksums match.")
print("Training images:", len(training))
print("Validation images:", len(validation))
print("Shared CNN and batch loader imported.")
print("Split checks passed. Training has not started.")

Dataset checksums match.
Training images: 23873
Validation images: 6103
Shared CNN and batch loader imported.
Split checks passed. Training has not started.


In [3]:
from datetime import datetime, timezone
import json

tf.keras.utils.set_random_seed(42)

baseline_train_ds = make_dataset(
    training,
    image_dir=dataset_dir / "train",
    batch_size=32,
    training_mode=True,
    seed=42,
)

baseline_val_ds = make_dataset(
    validation,
    image_dir=dataset_dir / "train",
    batch_size=32,
    training_mode=False,
)

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
baseline_dir = (
    Path("/kaggle/working/artifacts/baseline_runs") / run_id
)
baseline_dir.mkdir(parents=True, exist_ok=False)

with tf.device("/GPU:0"):
    baseline_model = build_model(learning_rate=0.001)

assert baseline_model.count_params() == 101665

baseline_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(baseline_dir / "best_baseline.keras"),
        monitor="val_loss",
        mode="min",
        save_best_only=True,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=5,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
    ),
    tf.keras.callbacks.CSVLogger(
        str(baseline_dir / "training_history.csv")
    ),
    tf.keras.callbacks.BackupAndRestore(
        backup_dir=str(baseline_dir / "recovery"),
        delete_checkpoint=False,
    ),
]

baseline_config = {
    "tensorflow_version": tf.__version__,
    "seed": 42,
    "max_epochs": 30,
    "batch_size": 32,
    "learning_rate": 0.001,
    "training_images": len(training),
    "validation_images": len(validation),
    "augmentation": False,
    "class_weights": None,
    "checkpoint_monitor": "val_loss",
    "threshold": 0.5,
    "comparison_type": "exploratory_single_seed_validation",
}

(baseline_dir / "baseline_config.json").write_text(
    json.dumps(baseline_config, indent=2),
    encoding="utf-8",
)

print("Baseline parameters:", baseline_model.count_params())
print("Baseline directory:", baseline_dir)
print("Augmentation: disabled")
print("Class weights: disabled")
print("Baseline configured. Training has not started.")

I0000 00:00:1792431166.266193      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1792431166.269874      49 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Baseline parameters: 101665
Baseline directory: /kaggle/working/artifacts/baseline_runs/20261019T173248_318248Z
Augmentation: disabled
Class weights: disabled
Baseline configured. Training has not started.


In [4]:
import time

baseline_start = time.perf_counter()

baseline_history = baseline_model.fit(
    baseline_train_ds,
    validation_data=baseline_val_ds,
    epochs=30,
    callbacks=baseline_callbacks,
    verbose=1,
)

baseline_seconds = time.perf_counter() - baseline_start

print("Completed baseline epochs:", len(baseline_history.epoch))
print("Training minutes:", round(baseline_seconds / 60, 2))
print("Best baseline:", baseline_dir / "best_baseline.keras")

Epoch 1/30
  2/747 ━━━━━━━━━━━━━━━━━━━━ 55s 75ms/step - accuracy: 0.4531 - auc: 0.5224 - loss: 1.0193 - precision: 0.4487 - recall: 1.0000   

I0000 00:00:1792431294.695571     145 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


747/747 ━━━━━━━━━━━━━━━━━━━━ 306s 410ms/step - accuracy: 0.9526 - auc: 0.9876 - loss: 0.1345 - precision: 0.9586 - recall: 0.9519 - val_accuracy: 0.4603 - val_auc: 0.7095 - val_loss: 3.0726 - val_precision: 1.0000 - val_recall: 0.0018 - learning_rate: 0.0010
Epoch 5/30
747/747 ━━━━━━━━━━━━━━━━━━━━ 316s 423ms/step - accuracy: 0.9584 - auc: 0.9910 - loss: 0.1155 - precision: 0.9638 - recall: 0.9577 - val_accuracy: 0.9644 - val_auc: 0.9933 - val_loss: 0.1011 - val_precision: 0.9678 - val_recall: 0.9664 - learning_rate: 5.0000e-04
Epoch 6/30
747/747 ━━━━━━━━━━━━━━━━━━━━ 314s 421ms/step - accuracy: 0.9646 - auc: 0.9943 - loss: 0.0931 - precision: 0.9661 - recall: 0.9673 - val_accuracy: 0.8035 - val_auc: 0.9141 - val_loss: 0.4752 - val_precision: 0.9023 - val_recall: 0.7139 - learning_rate: 2.5000e-04
Epoch 9/30
747/747 ━━━━━━━━━━━━━━━━━━━━ 309s 413ms/step - accuracy: 0.9718 - auc: 0.9960 - loss: 0.0773 - precision: 0.9710 - recall: 0.9761 - val_accuracy: 0.7662 - val_auc: 0.9814 - val_loss:

In [5]:
import pandas as pd

baseline_log = pd.read_csv(
    baseline_dir / "training_history.csv"
)

print("History object epochs:", [e + 1 for e in baseline_history.epoch])
print("CSV epochs:", (baseline_log["epoch"] + 1).tolist())
print("CSV rows:", len(baseline_log))

History object epochs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
CSV epochs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
CSV rows: 10


In [6]:
# Display metrics for every completed epoch.
epoch_results = baseline_log.copy()
epoch_results["epoch"] = epoch_results["epoch"] + 1

columns = [
    "epoch",
    "accuracy",
    "loss",
    "precision",
    "recall",
    "auc",
    "val_accuracy",
    "val_loss",
    "val_precision",
    "val_recall",
    "val_auc",
    "learning_rate",
]

available_columns = [
    column for column in columns
    if column in epoch_results.columns
]

print(
    epoch_results[available_columns].to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}",
    )
)

 epoch  accuracy   loss  precision  recall    auc  val_accuracy  val_loss  val_precision  val_recall  val_auc  learning_rate
     1    0.8817 0.2912     0.9077  0.8655 0.9478        0.5627    1.1487         0.5530      0.9982   0.9428         0.0010
     2    0.9391 0.1777     0.9504  0.9342 0.9790        0.8896    0.2722         0.9910      0.8030   0.9869         0.0010
     3    0.9441 0.1575     0.9529  0.9413 0.9833        0.7233    0.9588         0.6648      0.9845   0.9223         0.0010
     4    0.9526 0.1345     0.9586  0.9519 0.9876        0.4603    3.0726         1.0000      0.0018   0.7095         0.0010
     5    0.9584 0.1155     0.9638  0.9577 0.9910        0.9644    0.1011         0.9678      0.9664   0.9933         0.0005
     6    0.9615 0.1061     0.9650  0.9626 0.9925        0.7829    0.5364         0.7148      0.9958   0.9753         0.0005
     7    0.9601 0.1083     0.9639  0.9609 0.9922        0.8984    0.2641         0.9748      0.8336   0.9812         0.0005


In [7]:
import shutil

for filename in [
    "best_baseline.keras",
    "training_history.csv",
    "baseline_config.json",
]:
    path = baseline_dir / filename
    if not path.is_file():
        raise FileNotFoundError(path)

backup_path = shutil.make_archive(
    base_name=str(
        baseline_dir.parent / f"{baseline_dir.name}_backup"
    ),
    format="zip",
    root_dir=baseline_dir.parent,
    base_dir=baseline_dir.name,
)

print("Baseline backup:", backup_path)

Baseline backup: /kaggle/working/artifacts/baseline_runs/20261019T173248_318248Z_backup.zip


In [ ]:
import hashlib
import time

original_path = find_one("best_model.keras")
baseline_path = baseline_dir / "best_baseline.keras"

expected_original_hash = (
    "806d01759ea3f389d1b052b5f2a5ae5bb6e9bd0d3a4931cb328726c0227530c9"
)

actual_hash = hashlib.sha256(original_path.read_bytes()).hexdigest()
assert actual_hash == expected_original_hash, (
    "Original checkpoint checksum mismatch."
)

assert baseline_path.is_file(), "Baseline checkpoint is missing."

print("1. Loading original checkpoint...", flush=True)
start = time.perf_counter()

with tf.device("/CPU:0"):
    comparison_original = tf.keras.models.load_model(
        original_path,
        compile=False,
    )

print("2. Loading best baseline checkpoint...", flush=True)

with tf.device("/CPU:0"):
    comparison_baseline = tf.keras.models.load_model(
        baseline_path,
        compile=False,
    )

for model in [comparison_original, comparison_baseline]:
    assert model.input_shape == (None, 224, 224, 1)
    assert model.output_shape == (None, 1)

print("Both checkpoints loaded.")
print("Load time (seconds):", round(time.perf_counter() - start, 2))
print("Original model and threshold remain unchanged.")

In [1]:
from pathlib import Path
import hashlib
import time

print("1. Checking exact checkpoint paths...", flush=True)

original_path = Path(
    "/kaggle/input/datasets/mamtakumaridxcv/"
    "covidx-trained-model-backup/"
    "20261018T170010_055087Z/best_model.keras"
)

baseline_path = Path(
    "/kaggle/input/datasets/mamtakumaridxcv/"
    "covidx-baseline-model-backup/"
    "20261019T173248_318248Z/best_baseline.keras"
)

for path in [original_path, baseline_path]:
    if not path.is_file():
        raise FileNotFoundError(path)
    print(path.name, "bytes:", path.stat().st_size, flush=True)

print("2. Verifying original checkpoint...", flush=True)

expected_hash = (
    "806d01759ea3f389d1b052b5f2a5ae5bb6e9bd0d3a4931cb328726c0227530c9"
)
assert hashlib.sha256(original_path.read_bytes()).hexdigest() == expected_hash

print("3. Importing TensorFlow...", flush=True)
import tensorflow as tf

print("4. Loading original model on CPU...", flush=True)
start = time.perf_counter()

with tf.device("/CPU:0"):
    comparison_original = tf.keras.models.load_model(
        original_path, compile=False
    )

print("5. Loading baseline model on CPU...", flush=True)

with tf.device("/CPU:0"):
    comparison_baseline = tf.keras.models.load_model(
        baseline_path, compile=False
    )

for model in [comparison_original, comparison_baseline]:
    assert model.input_shape == (None, 224, 224, 1)
    assert model.output_shape == (None, 1)

print("Both checkpoints loaded successfully.", flush=True)
print("Load time (seconds):", round(time.perf_counter() - start, 2))

1. Checking exact checkpoint paths...
best_model.keras bytes: 1291457
best_baseline.keras bytes: 1291415
2. Verifying original checkpoint...
3. Importing TensorFlow...
4. Loading original model on CPU...


I0000 00:00:1792442737.151338      48 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1792442737.158610      48 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


5. Loading baseline model on CPU...
Both checkpoints loaded successfully.
Load time (seconds): 4.61


In [2]:
import sys
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

validation_path = Path(
    "/kaggle/input/datasets/mamtakumaridxcv/"
    "covidx-audit-backup/artifacts/manifests/validation.csv"
)

train_label_path = Path("/kaggle/input/covidx-cxr2/train.txt")

module_path = Path(
    "/kaggle/input/datasets/mamtakumaridxcv/"
    "chest-xray-project-source/"
    "Chest-X-Ray-Diagnosis-using-Deep-Learning--main/"
    "src/covid_xray/data.py"
)

for path in [validation_path, train_label_path, module_path]:
    if not path.is_file():
        raise FileNotFoundError(path)

expected_label_hash = (
    "52af9a4fdc769396bc5ce530872cf0d51bce"
    "8fd7e25579b333007d4ebee6a6ba"
)

assert hashlib.sha256(
    train_label_path.read_bytes()
).hexdigest() == expected_label_hash

validation = pd.read_csv(
    validation_path,
    dtype=str,
    keep_default_na=False,
)

assert len(validation) == 6103
assert validation["filename"].is_unique
assert validation["sha256"].is_unique
assert validation["label"].value_counts().to_dict() == {
    "positive": 3300,
    "negative": 2803,
}

image_dir = train_label_path.parent / "train"

missing = [
    filename for filename in validation["filename"]
    if not (image_dir / filename).is_file()
]
assert not missing, f"Missing validation images: {missing[:5]}"

sys.path.insert(0, str(module_path.parents[1]))

from covid_xray.data import make_dataset

comparison_val_ds = make_dataset(
    validation,
    image_dir=image_dir,
    batch_size=32,
    training_mode=False,
)

# Verify one batch before full comparison.
images, labels = next(iter(comparison_val_ds))

for name, model in [
    ("original", comparison_original),
    ("baseline", comparison_baseline),
]:
    with tf.device("/CPU:0"):
        scores = model(images, training=False).numpy()

    assert scores.shape == (32, 1)
    assert np.isfinite(scores).all()
    assert ((scores >= 0) & (scores <= 1)).all()

    print(f"{name}: real validation batch passed.", flush=True)

print("Validation images:", len(validation))
print("Validation batches:", int(
    tf.data.experimental.cardinality(comparison_val_ds)
))
print("Comparison setup passed. No training performed.")

original: real validation batch passed.
baseline: real validation batch passed.
Validation images: 6103
Validation batches: 191
Comparison setup passed. No training performed.


In [3]:
import json
import time
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,)

true_batches = []
score_batches = {"original": [], "baseline": []}
start = time.perf_counter()

models = {
    "original": comparison_original,
    "baseline": comparison_baseline,}

for number, (images, labels) in enumerate(comparison_val_ds, start=1):
    true_batches.append(labels.numpy().ravel())

    for name, model in models.items():
        with tf.device("/CPU:0"):
            scores = model(images, training=False).numpy().ravel()

        assert np.isfinite(scores).all()
        assert ((scores >= 0) & (scores <= 1)).all()
        score_batches[name].append(scores)

    if number % 20 == 0:
        print(f"Compared {number}/191 batches", flush=True)

y_true = np.concatenate(true_batches).astype(int)
expected_labels = validation["label"].map(
    {"negative": 0, "positive": 1}
).to_numpy(dtype=int)

np.testing.assert_array_equal(y_true, expected_labels)

rows = []

for name in models:
    scores = np.concatenate(score_batches[name])
    assert len(scores) == 6103

    predictions = (scores > 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_true, predictions, labels=[0, 1]
    ).ravel()

    rows.append({
        "model": name,
        "images": len(y_true),
        "threshold": 0.5,
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "f1": f1_score(y_true, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_true, scores),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),})

comparison_report = pd.DataFrame(rows).set_index("model")

baseline_fn = int(comparison_report.loc["baseline", "fn"])
original_fn = int(comparison_report.loc["original", "fn"])

relative_fn_reduction = (
    100 * (baseline_fn - original_fn) / baseline_fn
    if baseline_fn > 0 else None)

results_dir = Path("/kaggle/working/artifacts/comparison")
results_dir.mkdir(parents=True, exist_ok=True)

comparison_report.to_csv(
    results_dir / "validation_baseline_comparison.csv")

summary = {
    "evaluation_split": "validation",
    "comparison_type": "exploratory_single_seed",
    "baseline_false_negatives": baseline_fn,
    "original_false_negatives": original_fn,
    "relative_fn_reduction_percent": relative_fn_reduction,
    "seconds": round(time.perf_counter() - start, 2),}

(results_dir / "comparison_summary.json").write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",)

print("\nValidation comparison:")
print(comparison_report.to_string())
print("\nSummary:")
print(json.dumps(summary, indent=2))

Compared 20/191 batches
Compared 40/191 batches
Compared 60/191 batches
Compared 80/191 batches
Compared 100/191 batches
Compared 120/191 batches
Compared 140/191 batches
Compared 160/191 batches
Compared 180/191 batches

Validation comparison:
          images  threshold  accuracy  precision    recall        f1   roc_auc    tn   fp   fn    tp
model                                                                                               
original    6103        0.5  0.899066   0.923879  0.886364  0.904732  0.957182  2562  241  375  2925
baseline    6103        0.5  0.964444   0.967830  0.966364  0.967096  0.993384  2697  106  111  3189

Summary:
{
  "evaluation_split": "validation",
  "comparison_type": "exploratory_single_seed",
  "baseline_false_negatives": 111,
  "original_false_negatives": 375,
  "relative_fn_reduction_percent": -237.83783783783784,
  "seconds": 748.49
}


## Exploratory Baseline Comparison

Both saved checkpoints were evaluated on the same 6,103-image
validation split using the same preprocessing and a fixed threshold
of 0.5.

| Metric | Original: augmentation + class weights | Baseline: neither |
| --- | ---: | ---: |
| Accuracy | 89.91% | 96.44% |
| Positive recall | 88.64% | 96.64% |
| Positive F1 | 90.47% | 96.71% |
| False negatives | 375 | 111 |

The original model produced 264 more false negatives than the
baseline. This experiment does not support a claim of 12%
false-negative reduction.

This is an exploratory, single-seed validation comparison.
Validation was also used for checkpoint selection. Random-state
history was not controlled identically across runs, and augmentation
and class weighting were changed together. Their individual effects
cannot be determined from this experiment.

Baseline performance on the official test split has not been evaluated.
The existing application checkpoint remains unchanged.